# Part 2: Constructing Exogenous Instruments and Trade Weights

**Objective:**  
To isolate the causal impact of international sanctions, we must address the fundamental endogeneity of both the imposition of sanctions and the bilateral trade volumes between the sender and the target.

In this notebook, we construct two crucial exogenous components for our 2SLS identification strategy:
1. **Political Distance (UNGA Voting):** A strictly exogenous predictor for the probability of a sanction being imposed (First Stage).
2. **Gravity-Predicted Trade Weights:** Actual trade flows decline mechanically when sanctions are imposed. To measure a target's true economic vulnerability without endogeneity bias, we estimate a structural gravity model using the Poisson Pseudo-Maximum Likelihood estimator (Santos Silva & Tenreyro, 2006). The predicted trade flows are then used to weight the sanction intensity.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from itertools import combinations
import time
import os
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

# Configuration: File paths
CLEANED_DIR = 'data/cleaned/'
RAW_DIR = 'data/raw/'

UNGA_FILE = os.path.join(RAW_DIR, '2025_9_19_ga_voting.csv')
GEODIST_FILE = os.path.join(CLEANED_DIR, 'cleaned_geodist_dyadic.csv')
TRADE_FILE = os.path.join(CLEANED_DIR, 'cleaned_trade_dyadic.csv')
SIPRI_FILE = os.path.join(CLEANED_DIR, 'cleaned_sipri_country_year.csv')

## 1. Political Distance: UN General Assembly Voting
Following the existing literature on international political economy, we measure the political misalignment between two states as the mean absolute difference in their voting choices at the UN General Assembly. This metric serves as an exogenous(!) predictor of a sender's propensity to sanction a target.

In [2]:
def calculate_unga_vote_distance(filepath):
    """
    Loads UNGA voting records, encodes votes numerically, and calculates the
    mean absolute difference in voting choices for all country dyads per year.
    """
    print("Loading and preprocessing UNGA voting data...")
    # Load specific columns to optimize memory usage
    df_unga = pd.read_csv(filepath, usecols=['ms_code', 'date', 'undl_id', 'ms_vote'])
    df_unga = df_unga.rename(columns={'ms_code': 'iso3'})

    # Extract year
    df_unga['year'] = pd.to_datetime(df_unga['date'], errors='coerce').dt.year

    # Map categorical votes to numeric scores (Y=1.0, N=0.0, Abstain/Absent=0.5)
    vote_mapping = {'Y': 1.0, 'N': 0.0, 'X': 0.5, 'A': 0.5}
    df_unga['vote_score'] = df_unga['ms_vote'].map(vote_mapping)
    df_unga = df_unga.dropna(subset=['iso3', 'year', 'undl_id', 'vote_score'])

    print("Calculating dyadic voting distance (this may take a few minutes)...")
    start_time = time.time()

    # Pivot table: rows = (year, resolution_id), columns = iso3, values = vote_score
    df_pivot = df_unga.pivot_table(index=['year', 'undl_id'], columns='iso3', values='vote_score')

    all_dyads = []
    grouped_by_year = df_pivot.groupby(level='year')

    for year, year_df in grouped_by_year:
        countries = year_df.columns
        country_pairs = list(combinations(countries, 2))

        for c1, c2 in country_pairs:
            # Keep only resolutions where both countries voted
            votes = year_df[[c1, c2]].dropna()
            if not votes.empty:
                # Mean absolute difference
                distance = np.mean(np.abs(votes[c1] - votes[c2]))
                all_dyads.append({'iso3_i': c1, 'iso3_j': c2, 'year': int(year), 'vote_distance': distance})

    df_distance = pd.DataFrame(all_dyads)
    print(f"Dyadic distance calculation completed in {time.time() - start_time:.2f} seconds.")

    return df_distance

# Smart bypass to save 36 minutes of computation
unga_dist_path = os.path.join(CLEANED_DIR, 'cleaned_unga_distance_dyadic.csv')

if os.path.exists(unga_dist_path):
    print("Pre-calculated UNGA distance file found! Loading it in 1 second...")
    unga_distance_df = pd.read_csv(unga_dist_path)
else:
    print("Pre-calculated file not found. Running calculations (may take ~35 mins)...")
    unga_distance_df = calculate_unga_vote_distance(UNGA_FILE)
    unga_distance_df.to_csv(unga_dist_path, index=False)

unga_distance_df.head()

Pre-calculated UNGA distance file found! Loading it in 1 second...


,iso3_i,iso3_j,year,vote_distance,Unnamed: 4
0,BLR,UKR,1946,0.0,ЛОЖЬ
1,IRQ,SYR,1946,0.0,ЛОЖЬ
2,SUN,YUG,1946,0.0,ЛОЖЬ
3,AUS,ZAF,1947,0.0,ЛОЖЬ
4,BLR,CSK,1947,0.0,ЛОЖЬ


## 2. Structural Gravity Model (PPML) and Exogenous Trade Weights
We estimate bilateral trade flows dictated strictly by exogenous geographic and historical determinants (GDP, distance, contiguity, language, colonial links).

By using the PPML estimator, we robustly handle zero trade flows and heteroskedasticity. The predicted shares derived from this model represent the "natural" level of structural trade dependence, entirely purged of contemporaneous political shocks or actual sanction policies.

In [3]:
def estimate_gravity_weights():
    """
    Estimates the structural gravity model using PPML for both export and import
    directions. Generates exogenous trade exposure weights for the sanction intensity.
    """
    print("--- Estimating Gravity Model (PPML) ---")
    start_time = time.time()

    # Load previously cleaned datasets
    trade_df = pd.read_csv(TRADE_FILE)
    geodist_df = pd.read_csv(GEODIST_FILE)
    gdp_df = pd.read_csv(SIPRI_FILE)[['iso3', 'year', 'gdp_const_usd']].dropna()

    # Standardize ISO codes and years
    for df in [trade_df, geodist_df, gdp_df]:
        for col in df.columns:
            if 'iso3' in col:
                df[col] = df[col].astype(str).str.strip().str.upper()

    trade_df['year'] = pd.to_numeric(trade_df['year'], errors='coerce').dropna().astype(int)
    gdp_df['year'] = pd.to_numeric(gdp_df['year'], errors='coerce').dropna().astype(int)

    # Merge dyadic components
    grav_df = pd.merge(trade_df, geodist_df, on=['iso3_i', 'iso3_j'], how='inner')
    grav_df = pd.merge(grav_df, gdp_df.rename(columns={'iso3': 'iso3_i', 'gdp_const_usd': 'gdp_i'}), on=['iso3_i', 'year'], how='inner')
    grav_df = pd.merge(grav_df, gdp_df.rename(columns={'iso3': 'iso3_j', 'gdp_const_usd': 'gdp_j'}), on=['iso3_j', 'year'], how='inner')

    # Clip negative values (COW data anomalies) and log-transform predictors
    grav_df['trade_flow_ji'] = grav_df['trade_flow_ji'].clip(lower=0)
    grav_df['trade_flow_ij'] = grav_df['trade_flow_ij'].clip(lower=0)

    grav_df['ln_gdp_i'] = np.log(grav_df['gdp_i'])
    grav_df['ln_gdp_j'] = np.log(grav_df['gdp_j'])
    grav_df['ln_dist'] = np.log(grav_df['distance'].replace(0, 1))

    # Standard gravity equation predictors
    gravity_formula = " ~ ln_gdp_i + ln_gdp_j + ln_dist + contiguity + common_language + colonial_link"
    vars_model = ['ln_gdp_i', 'ln_gdp_j', 'ln_dist', 'contiguity', 'common_language', 'colonial_link']

    print(f"Estimating PPML on {len(grav_df)} dyad-year observations...")

    # 1. Target's IMPORTS from Sender (trade_flow_ij).
    # Used to measure vulnerability to Sender's EXPORT sanctions.
    print("  -> Fitting model for Target's Import Dependence (flow i -> j)...")
    subset_imp = grav_df.dropna(subset=vars_model + ['trade_flow_ij']).copy()
    model_imp = smf.glm(formula="trade_flow_ij" + gravity_formula,
                        data=subset_imp, family=sm.families.Poisson()).fit()
    subset_imp.loc[:, 'pred_target_imports'] = model_imp.predict()

    # 2. Target's EXPORTS to Sender (trade_flow_ji).
    # Used to measure vulnerability to Sender's IMPORT sanctions.
    print("  -> Fitting model for Target's Export Dependence (flow j -> i)...")
    subset_exp = grav_df.dropna(subset=vars_model + ['trade_flow_ji']).copy()
    model_exp = smf.glm(formula="trade_flow_ji" + gravity_formula,
                        data=subset_exp, family=sm.families.Poisson()).fit()
    subset_exp.loc[:, 'pred_target_exports'] = model_exp.predict()

    # Merge predictions back together
    res = pd.merge(subset_imp[['iso3_i', 'iso3_j', 'year', 'pred_target_imports']],
                   subset_exp[['iso3_i', 'iso3_j', 'year', 'pred_target_exports']],
                   on=['iso3_i', 'iso3_j', 'year'], how='outer').fillna(0)

    # Calculate Total Predicted Trade for Target j
    total = res.groupby(['iso3_j', 'year'])[['pred_target_imports', 'pred_target_exports']].sum().reset_index()
    total.rename(columns={'pred_target_imports': 'total_imports_j', 'pred_target_exports': 'total_exports_j'}, inplace=True)

    res = pd.merge(res, total, on=['iso3_j', 'year'], how='left')

    # Calculate Final Exogenous Weights
    # If Sender i applies Export Sanctions, Target j suffers based on its import dependence on i
    res['weight_export'] = (res['pred_target_imports'] / res['total_imports_j'].replace(0, np.nan)).fillna(0)

    # If Sender i applies Import Sanctions, Target j suffers based on its export dependence on i
    res['weight_import'] = (res['pred_target_exports'] / res['total_exports_j'].replace(0, np.nan)).fillna(0)

    print(f"Gravity weights calculated in {time.time() - start_time:.2f} seconds.")
    return res[['iso3_i', 'iso3_j', 'year', 'weight_export', 'weight_import']]

# Execute and display snippet
gravity_weights_df = estimate_gravity_weights()
gravity_weights_df.to_csv(os.path.join(CLEANED_DIR, 'gravity_trade_weights.csv'), index=False)
gravity_weights_df.head()

--- Estimating Gravity Model (PPML) ---


Estimating PPML on 211480 dyad-year observations...
  -> Fitting model for Target's Import Dependence (flow i -> j)...


  -> Fitting model for Target's Export Dependence (flow j -> i)...


Gravity weights calculated in 4.99 seconds.


,iso3_i,iso3_j,year,weight_export,weight_import
0,AFG,AUS,1970,0.001613,0.000802
1,AFG,AUS,1973,0.001381,0.000677
2,AFG,AUS,1974,0.001464,0.000737
3,AFG,AUS,1975,0.001477,0.000736
4,AFG,AUS,1976,0.001461,0.000736
